# uMaxPro and MaxPro Study (using joblib)

This notebooks performs a raxpy experiment on raxpy itself comparing uMaxPro designs to MaxPro designs.

The joblib adapter is used to parallize the experiment trials.

In [ ]:
from typing import Annotated, Tuple
import time

import raxpy
import raxpy.spaces.dimensions as dims
import raxpy.adapters.joblib as jl
import raxpy.does.factorial as factorial
import raxpy.annotations.function_spec as fs
import raxpy.does.plots as dplots
import raxpy.does.measure as dm

import functools

In [ ]:
strategies = {
    "MaxPro":  functools.partial(raxpy.design_experiment, optimize_projections="MaxPro"),
    "uMaxPro": functools.partial(raxpy.design_experiment, optimize_projections="uMaxPro"),
}

spaces = {
    "3-Flat":
        raxpy.spaces.InputSpace(
            dimensions=[
                dims.Float(id="x1", lb=0.0, ub=1.0),
                dims.Float(id="x2", lb=0.0, ub=1.0),
                dims.Float(id="x3", lb=0.0, ub=1.0),
            ]
        ),
    "3-Mixed":
        raxpy.spaces.InputSpace(
            dimensions=[
                dims.Float(id="x1", lb=0.0, ub=1.0),
                dims.Float(id="x2", lb=0.0, ub=1.0, nullable=True, portion_null=0.25),
                dims.Float(id="x3", lb=0.0, ub=1.0, nullable=True, portion_null=0.25),
            ]
        )
}

def design_experiment(
    seed:Annotated[int, raxpy.Integer(lb=1, ub=10)],
    space_id:Annotated[str, raxpy.Categorical(value_set=("3-Flat", "3-Mixed"))],
    design_size:Annotated[int, raxpy.Integer(value_set=(1, 2, 3))],
    design_mode:Annotated[str, raxpy.Categorical(value_set=("MaxPro", "uMaxPro"))],
) -> Tuple[
    float, object
]:
    input_space = spaces[space_id]
    n_points = input_space.count_dimensions() * design_size * 2
    
    start_time = time.time()

    doe = strategies[design_mode](input_space, n_points=n_points, seed=seed)
    
    duration = time.time() - start_time
    
    return (
        duration,
        doe,
    )
    

In [ ]:
input_space = fs.extract_input_space(design_experiment)
input_space

In [ ]:
full_doe = factorial.generate_full_factorial_design(input_space)
full_doe

In [ ]:
inputs, outputs = jl.perform_experiment(full_doe, design_experiment, n_jobs=8)

In [ ]:
metrics = [dm.measure_with_all_metrics(o[1]) for o in outputs]

In [ ]:
metrics[0]

In [ ]:
inputs[0]

In [ ]:
inputs[1]

## Visualize Pair Designs

In [ ]:
from ipywidgets import interact, widgets

In [ ]:
def compare_except_key(d1, d2, ignore_key):
    # Ensure they have the same structure/keys first
    if set(d1) == set(d2):
        # If the only difference is the ignored key, they might have different lengths
        # Adjusting the keys check to omit the ignored key
        return set(d1).difference([ignore_key]) == set(d2).difference([ignore_key]) and \
               all(d1[k] == d2[k] for k in d1 if k != ignore_key)
    return False

# 1. Define your plotting function with parameters
def plot_doe_pair(i):
    out1 = widgets.Output()
    out2 = widgets.Output()
    with out1:
        dplots.plot_scatterplot_matrix(outputs[i][1], title=f"Run {i}, Metric: {metrics[i].measurements['max_whole_min_point_distance']}: " + inputs[i]["design_mode"])

    # find pair 
    input_values = inputs[i]

    for j in range(len(inputs)):
        if j != i and compare_except_key(inputs[j], input_values, 'design_mode'):
            with out2:
                dplots.plot_scatterplot_matrix(outputs[j][1], title=f"Run {j}, Metric: {metrics[j].measurements['max_whole_min_point_distance']}: " +  inputs[j]["design_mode"])
            break
    layout = widgets.HBox([out1, out2])
    display(layout)

    


# 2. Use 'interact' to automatically create sliders for the arguments
interact(plot_doe_pair, i=(0, len(outputs) - 1, 1))



## DoE Metric histograms

In [ ]:
import matplotlib.pyplot as plt

plt.hist([metrics[i].measurements['max_whole_min_point_distance'] for i in range(len(metrics)) if inputs[i]["design_mode"] == "MaxPro"], bins=20, alpha=0.5, label='MaxPro')
plt.hist([metrics[i].measurements['max_whole_min_point_distance'] for i in range(len(metrics)) if inputs[i]["design_mode"] == "uMaxPro"], bins=20, alpha=0.5, label='uMaxPro')
plt.legend()


In [ ]:
import matplotlib.pyplot as plt

plt.hist([metrics[i].measurements['max_pro'] for i in range(len(metrics)) if inputs[i]["design_mode"] == "MaxPro"], bins=20, alpha=0.5, label='MaxPro')
plt.hist([metrics[i].measurements['max_pro'] for i in range(len(metrics)) if inputs[i]["design_mode"] == "uMaxPro"], bins=20, alpha=0.5, label='uMaxPro')
plt.legend()

In [ ]:
import matplotlib.pyplot as plt

plt.hist([metrics[i].measurements['star_discrepancy'] for i in range(len(metrics)) if inputs[i]["design_mode"] == "MaxPro"], bins=20, alpha=0.5, label='MaxPro')
plt.hist([metrics[i].measurements['star_discrepancy'] for i in range(len(metrics)) if inputs[i]["design_mode"] == "uMaxPro"], bins=20, alpha=0.5, label='uMaxPro')
plt.legend()

## DoE Paired Statistical Tests

In [ ]:
# perform paired comparison of metrics for each pair of runs with the same input values but different design modes
def derive_paired_values(metric_name):
    paired_values = []
    for i in range(len(inputs)):
        design_mode_i = inputs[i]['design_mode']

        for j in range(i + 1, len(inputs)):
            if compare_except_key(inputs[i], inputs[j], 'design_mode'):
                if design_mode_i == 'MaxPro':
                    paired_values.append((metrics[i].measurements[metric_name], metrics[j].measurements[metric_name]))
                else:   
                    paired_values.append((metrics[j].measurements[metric_name], metrics[i].measurements[metric_name]))
                break
    return paired_values

In [ ]:
# compute statistics
paired_max_whole_min_point_distance = derive_paired_values('max_whole_min_point_distance')

import scipy.stats as stats

stats.ttest_rel([v[0] for v in paired_max_whole_min_point_distance], [v[1] for v in paired_max_whole_min_point_distance])

In [ ]:
# compute statistics
paired_max_pro = derive_paired_values('max_pro')

import scipy.stats as stats

stats.ttest_rel([v[0] for v in paired_max_pro], [v[1] for v in paired_max_pro])

In [ ]:
# compute statistics
paired_star_discrepancy = derive_paired_values('star_discrepancy')

import scipy.stats as stats

stats.ttest_rel([v[0] for v in paired_star_discrepancy], [v[1] for v in paired_star_discrepancy])

In [ ]:
# compute statistics
paired_weighted_mipd= derive_paired_values('weighted_mipd')

import scipy.stats as stats

stats.ttest_rel([v[0] for v in paired_weighted_mipd], [v[1] for v in paired_weighted_mipd])